# FCI Procurement — MSP vs MSV analysis
Loads the repo's verified datasets straight from GitHub and reproduces the core analysis:
procurement trends, the MSP coverage gap, stocks vs buffer norms, and offtake channels.

- Repo: https://github.com/herrrickshaw/FCI-warehouse-MSPvsMSV
- Data provenance: [`data/MANIFEST.md`](https://github.com/herrrickshaw/FCI-warehouse-MSPvsMSV/blob/main/data/MANIFEST.md)
- Units: lakh metric tonnes (LMT). Rice 2024-25 procurement is a derived full-season
  rice-equivalent (832.17 LMT paddy × 0.67 outturn) — see MANIFEST.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

RAW = "https://raw.githubusercontent.com/herrrickshaw/FCI-warehouse-MSPvsMSV/main/data/"
proc   = pd.read_csv(RAW + "procurement_by_year.csv")
pvp    = pd.read_csv(RAW + "production_vs_procurement.csv")
stocks = pd.read_csv(RAW + "stocks_vs_norms.csv", parse_dates=["date"])
off    = pd.read_csv(RAW + "offtake_by_scheme.csv")
stor   = pd.read_csv(RAW + "storage_capacity.csv")
eth    = pd.read_csv(RAW + "ethanol_allocation.csv")
print({n: df.shape for n, df in
       [("procurement",proc),("prod_vs_proc",pvp),("stocks",stocks),
        ("offtake",off),("storage",stor),("ethanol",eth)]})

## 1. The MSP coverage gap — procurement as % of production, latest full year per crop

In [ ]:
latest = (pvp.dropna(subset=["procurement_share_pct"])
            .sort_values("year").groupby("crop").tail(1)
            .query("crop not in ('coarsegrains','nigerseed')")
            .sort_values("procurement_share_pct"))
ax = latest.plot.barh(x="crop", y="procurement_share_pct", legend=False,
                      color="#2a78d6", figsize=(9, 6))
for i, (s, y) in enumerate(zip(latest.procurement_share_pct, latest.year)):
    ax.text(s + 0.5, i, f"{s}%  ({y})", va="center", fontsize=9, color="#52514e")
ax.set_xlabel("share of production procured (%)"); ax.set_ylabel("")
ax.set_title("MSP is announced for 23+ crops — volume shows up for two")
ax.spines[["top","right"]].set_visible(False)
plt.tight_layout(); plt.show()

## 2. Rice & wheat procurement, all-India (KMS/RMS 2010-11 → 2025-26)

In [ ]:
ai = proc[proc.state.isin(["ALL_INDIA","All India","ALL INDIA"])]
piv = ai.pivot_table(index="year", columns="crop", values="quantity_lmt", aggfunc="sum")
# full-season rice-equivalent for KMS 2024-25 (832.17 LMT paddy x 0.67)
if "rice" in piv and "2024-25" in piv.index: piv.loc["2024-25","rice"] = 557.6
ax = piv[["rice","wheat"]].plot(figsize=(9,4.5), color=["#2a78d6","#eb6834"], lw=2)
ax.set_ylabel("LMT"); ax.set_xlabel("")
ax.set_title("Procurement: rice ~2x wheat; 2022-23 heatwave crash visible in wheat")
ax.spines[["top","right"]].set_visible(False); plt.tight_layout(); plt.show()

## 3. Central pool stocks vs buffer norms (monthly, 2021-24)

In [ ]:
tot = stocks[stocks.commodity=="total"].set_index("date").sort_index()
ax = tot.stock_lmt.plot(figsize=(9,4.5), color="#2a78d6", lw=2, label="Total stock")
tot.buffer_norm_lmt.dropna().plot(ax=ax, style="o", color="#eb6834", label="Buffer norm")
ax.set_ylabel("LMT"); ax.set_xlabel(""); ax.legend()
ax.set_title("Stocks run far above the quarterly minimum norms")
ax.spines[["top","right"]].set_visible(False); plt.tight_layout(); plt.show()

## 4. Offtake by channel — where the grain goes

In [ ]:
o = off[(off.measure=="offtake") & (off.scheme!="TOTAL")].copy()
grp = {"NFSA_TPDS":"NFSA/TPDS","PMGKAY":"PMGKAY","PMGKAY_ANBP":"PMGKAY",
       "OMSS_Domestic":"OMSS(D)"}
o["channel"] = o.scheme.map(grp).fillna("Other welfare")
piv = o.pivot_table(index="year", columns="channel", values="quantity_lmt", aggfunc="sum")
cols = [c for c in ["NFSA/TPDS","PMGKAY","OMSS(D)","Other welfare"] if c in piv]
ax = piv[cols].plot.bar(stacked=True, figsize=(9,4.5),
        color=["#2a78d6","#eb6834","#1baf7a","#eda100"])
ax.set_ylabel("LMT (rice+wheat)"); ax.set_xlabel("")
ax.set_title("OMSS(D) = sales to processors/millers. 2024-25 is part-year (Nov-2024 cut-off)")
ax.spines[["top","right"]].set_visible(False); plt.tight_layout(); plt.show()

## 5. Storage — the physical ceiling on any volume guarantee
Covered storage ≈ 776.6 LMT (30.11.2024). IEG WP420: public storage caps procurement at
~30% of the ~261 MT marketed surplus of 14 major crops — the binding constraint on a
universal MSV. See the main report for the full argument.

In [ ]:
agencies = {"state_agencies_incl_SWC":"State agencies","FCI_owned":"FCI owned",
  "FCI_owned_silo":"FCI owned","hired_SWC":"Hired-SWC","hired_PEG":"Hired-PEG",
  "hired_CWC":"Hired-CWC","hired_PWS2010":"Hired-other","hired_state_govt":"Hired-other",
  "hired_private":"Hired-other","hired_silo":"Hired-other"}
s = stor[stor.agency.isin(agencies)].copy()
s["grp"] = s.agency.map(agencies)
tot = s.groupby("grp").capacity_lmt.sum().sort_values()
ax = tot.plot.barh(color="#2a78d6", figsize=(8,3.5))
ax.set_xlabel("LMT"); ax.set_ylabel("")
ax.set_title(f"Covered storage by agency — total {tot.sum():.0f} LMT")
ax.spines[["top","right"]].set_visible(False); plt.tight_layout(); plt.show()